In [1]:
import pandas as pd

In [13]:
analysis_df = pd.read_csv('./ecir_res/union_output_v2.csv')
single_df = pd.read_csv('./ecir_res/single_output_v2.csv')

In [3]:
gpp_df = analysis_df[analysis_df['Prediction Name']=='GPP']
rpp_df = analysis_df[analysis_df['Prediction Name']=='RPP']

In [4]:
from tools import fisher_test

def f(_x):
    _dict = {'0': 'QPP', '1': 'PerpC', '2': 'QualT5', '3': 'Readability'}
    output_list = [False, False, False, False]
    for i in _x[1:]:
        output_list[int(i)] = True
    return output_list

def significance(_n, _x, _b):
    _oa, _ob = fisher_test.compare_spearman_rhos(_n, _b, _x).values()
    return (_oa<0)&(_ob<0.05)

In [5]:
import copy
import itertools

y = 0

show_experiments = []
for k, retr in itertools.product([2, 3, 5, 7, 10], ['bm25', 'mt5', 'e5']):
    
    x = gpp_df.query('Use_Postgen==True').copy()
    n = x['Number of Queries'].values[0]
    x = x.reset_index(drop=True)
    x = x[x['QA Task']=='nq']
    x = x[x['Retriever']==retr]
    x = x[x['Top-Retrieved Docs']==k]
    identifier = f'{retr}%{k}'
    show_experiments.append(identifier)
    
    best_single = x[x.Combination_Number=='p0123']['Best Single Rho'].values[0]
    best_single_name = x[x.Combination_Number=='p0123']['Best Single Signal'].values[0]
    
    x_a = pd.DataFrame(x.Combination_Number.apply(f).tolist(), columns=['QPP', 'PerpC', 'Qual', 'Read'])
    x = pd.concat([x_a, x.reset_index(drop=True)], axis=1).drop(columns=['Prediction Name', 'QA Task', 'Top-Retrieved Docs', 'Number of Queries', \
                                                                     'Tau', 'Best Single Tau', 'Best Single Rho', 'Best Single Signal', \
                                                                     'Combination_Number', 'Use_Postgen', 'Retriever'])
    x = x.rename(columns={'Rho': identifier})
    best_row = pd.DataFrame([dict(zip(x.columns.tolist(), [False, False, False, False, best_single]))])
    x = pd.concat([best_row, x], ignore_index=True)
    x[f'Sig:{identifier}'] = x[identifier].apply(lambda _x: significance(n, _x, best_single))

    if(type(y)==int):
        y = copy.deepcopy(x)
    else:
        y = pd.concat([y, x[[identifier, f'Sig:{identifier}']].copy()], axis=1)
y

,QPP,PerpC,Qual,Read,bm25%2,Sig:bm25%2,mt5%2,Sig:mt5%2,e5%2,Sig:e5%2,...,mt5%7,Sig:mt5%7,e5%7,Sig:e5%7,bm25%10,Sig:bm25%10,mt5%10,Sig:mt5%10,e5%10,Sig:e5%10
0,False,False,False,False,0.303700,False,0.299200,False,0.287300,False,...,0.349400,False,0.342000,False,0.390800,False,0.391300,False,0.399900,False
1,True,False,False,False,0.350698,True,0.361572,True,0.391543,True,...,0.373732,False,0.385134,True,0.411471,False,0.407911,False,0.433580,True
2,False,True,False,False,0.312849,False,0.309987,False,0.317680,False,...,0.359599,False,0.352206,False,0.394980,False,0.396593,False,0.402362,False
3,False,False,True,False,0.309378,False,0.285096,False,0.300525,False,...,0.351783,False,0.339186,False,0.391094,False,0.393149,False,0.400937,False
4,False,False,False,True,0.317063,False,0.267533,False,0.292194,False,...,0.352539,False,0.340891,False,0.388622,False,0.390045,False,0.396099,False
5,True,True,False,False,0.365314,True,0.372608,True,0.398786,True,...,0.378353,False,0.389672,True,0.415224,False,0.412161,False,0.435148,True
6,True,False,True,False,0.359688,True,0.380139,True,0.392175,True,...,0.371798,False,0.382582,True,0.413426,False,0.406750,False,0.433436,True
7,True,False,False,True,0.357262,True,0.362839,True,0.391269,True,...,0.373024,False,0.382615,True,0.406496,False,0.407494,False,0.428679,True
8,False,True,True,False,0.313757,False,0.313220,False,0.318449,True,...,0.358061,False,0.348645,False,0.395887,False,0.397312,False,0.402138,False
9,False,True,False,True,0.324339,False,0.314272,False,0.317194,False,...,0.359085,False,0.350465,False,0.391941,False,0.392039,False,0.396945,False


In [6]:
from IPython.display import display, HTML
# Function to replace True/False with tick/blank
def tickmark(val):
    return "✔️" if val else ""

# Function to bold if Sig:e5%3 is True
def bold_sig(val, sig):
    return f"<b>{val:.6f}</b>" if sig else f"{val:.6f}"

# Apply formatting
df_display = y.copy()

# Replace first 4 columns with ticks
for col in ["QPP", "PerpC", "Qual", "Read"]:
    df_display[col] = df_display[col].apply(tickmark)

# Bold e5%3 where Sig:e5%3 is True
for _id in show_experiments:

    df_display[_id] = [
        bold_sig(v, s) for v, s in zip(y[_id], y[f"Sig:{_id}"])
    ]

    df_display = df_display.drop(columns=[f"Sig:{_id}"])

# Display with HTML rendering
df_display = df_display.style.hide(axis="index").to_html()
display(HTML(df_display))

QPP,PerpC,Qual,Read,bm25%2,mt5%2,e5%2,bm25%3,mt5%3,e5%3,bm25%5,mt5%5,e5%5,bm25%7,mt5%7,e5%7,bm25%10,mt5%10,e5%10
,,,,0.303700,0.299200,0.287300,0.308000,0.267600,0.292400,0.325600,0.322500,0.304800,0.385600,0.349400,0.342000,0.390800,0.391300,0.399900
✔️,,,,0.350698,0.361572,0.391543,0.365880,0.357803,0.381001,0.373276,0.363923,0.362266,0.412566,0.373732,0.385134,0.411471,0.407911,0.433580
,✔️,,,0.312849,0.309987,0.317680,0.316831,0.306201,0.316973,0.336941,0.342900,0.315715,0.391893,0.359599,0.352206,0.394980,0.396593,0.402362
,,✔️,,0.309378,0.285096,0.300525,0.317511,0.288985,0.302507,0.331931,0.331203,0.304382,0.387797,0.351783,0.339186,0.391094,0.393149,0.400937
,,,✔️,0.317063,0.267533,0.292194,0.315454,0.276763,0.293673,0.331016,0.324782,0.306509,0.382565,0.352539,0.340891,0.388622,0.390045,0.396099
✔️,✔️,,,0.365314,0.372608,0.398786,0.375908,0.365326,0.386401,0.380003,0.368836,0.364674,0.420406,0.378353,0.389672,0.415224,0.412161,0.435148
✔️,,✔️,,0.359688,0.380139,0.392175,0.367087,0.365122,0.380116,0.377207,0.366001,0.361367,0.416029,0.371798,0.382582,0.413426,0.406750,0.433436
✔️,,,✔️,0.357262,0.362839,0.391269,0.369089,0.361404,0.379617,0.375783,0.363910,0.360263,0.408892,0.373024,0.382615,0.406496,0.407494,0.428679
,✔️,✔️,,0.313757,0.313220,0.318449,0.318620,0.308629,0.314247,0.340856,0.344201,0.312215,0.394334,0.358061,0.348645,0.395887,0.397312,0.402138
,✔️,,✔️,0.324339,0.314272,0.317194,0.324953,0.314163,0.316080,0.342417,0.342326,0.315343,0.389694,0.359085,0.350465,0.391941,0.392039,0.396945


### For Table 1

In [16]:
qpps = ['nqc', 'maxScore', 'spatial', 'a_ratio', 'bertQPP']

single_df[single_df.k==2]

,k,retriever,task,target_metric,QPP_Method,Spearman,Kendall
0,2,bm25,nq,f1,nqc,0.0610,0.0467
1,2,bm25,nq,f1,maxScore,0.1609,0.1235
2,2,bm25,nq,f1,spatial,0.2166,0.1659
3,2,bm25,nq,f1,a_ratio,0.1353,0.1040
4,2,bm25,nq,f1,bertQPP,0.2079,0.1594
...,...,...,...,...,...,...,...
139,2,mt5,dl,utility,itg(perpC),0.0430,0.0335
140,2,mt5,dl,utility,max(docQual),0.1325,0.0909
141,2,mt5,dl,utility,min(docQual),0.1942,0.1413
142,2,mt5,dl,utility,avg(docQual),0.1869,0.1306
